# 💬 LangGraph Fundamentals: State, Nodes, Edges, Tools & Memory

Yeh notebook **LangGraph** ke fundamental building blocks ko systematically cover karta hai:
1. **Core Primitives**: State, Nodes, aur Edges (`START` / `END`).
2. **ReAct Tool-Calling Agent**: `@tool`, `bind_tools`, aur `ToolNode` integration.
3. **Memory Checkpointing**: `MemorySaver` aur `thread_id` se conversation state persistence.
4. **Streaming Modes**: Real-time token aur node updates stream karna.
5. **Human-in-the-Loop (HITL)**: Tool execution se pehle graph interrupt karna.

### 📦 1. Environment Setup & Model Initialization
- **Purpose**: API keys load karna aur ChatGroq model initialize karna.
- **Libraries**: `langchain`, `langgraph`, `dotenv`

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
model = init_chat_model(model="openai/gpt-oss-20b", model_provider="groq")
print("Model initialized successfully.")

### 🧱 2. State Definition with `add_messages` Reducer
- **Class**: `State(TypedDict)`
- **Field**: `messages: Annotated[list, add_messages]`
- **Concept**:
  - `TypedDict`: Graph state ka data contract define karta hai.
  - `add_messages`: Reducer function jo har node execution ke baad purani list overwrite karne ke bajay naye messages append karta hai.

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

print("State schema defined.")

### 🤖 3. Basic StateGraph Chatbot
- **Node Function**: `chatbot(state: State) -> dict`
- **Graph Flow**: `START ➔ chatbot ➔ END`
- **Purpose**: Minimal runnable graph create karke invoke aur stream test karna.

In [3]:
def chatbot(state: State):
    return {"messages": [model.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

basic_graph = graph_builder.compile()

# Invoke test
response = basic_graph.invoke({"messages": [HumanMessage(content="Hello! Who are you?")]})
print("Response:", response["messages"][-1].content)

### 🛠️ 4. Tool Creation & Binding (`@tool` & `ToolNode`)
- **Tool**: `multiply(a: int, b: int) -> int`
- **Purpose**: LLM ko deterministic function execution capability provide karna.
- **Components**:
  - `bind_tools`: Model ko available tools ka schema provide karta hai.
  - `ToolNode`: LangGraph ka prebuilt executor jo tool calls run karta hai.
  - `tools_condition`: Conditional edge jo check karti hai ki LLM ne tool call request kiya hai ya direct reply diya hai.

In [4]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b

tools = [multiply]
model_with_tools = model.bind_tools(tools)

def tool_calling_llm(state: State):
    return {"messages": [model_with_tools.invoke(state["messages"])]}

# Construct ReAct Tool Graph
tool_builder = StateGraph(State)
tool_builder.add_node("agent", tool_calling_llm)
tool_builder.add_node("tools", ToolNode(tools))

tool_builder.add_edge(START, "agent")
tool_builder.add_conditional_edges("agent", tools_condition)
tool_builder.add_edge("tools", "agent")

react_graph = tool_builder.compile()
print("ReAct Tool Graph compiled successfully.")

### 🧠 5. Conversation Memory & State Checkpointing (`MemorySaver`)
- **Class**: `MemorySaver`
- **Purpose**: Session state aur multi-turn dialogue ko thread ID ke sath persist karna.
- **Configuration**: `config = {"configurable": {"thread_id": "session_1"}}`

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
stateful_graph = tool_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "user_session_1"}}

# Turn 1: Introduce user
stateful_graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Mohit and I am studying AI agents.")]},
    config=config
)

# Turn 2: Query memory
turn2 = stateful_graph.invoke(
    {"messages": [HumanMessage(content="What is my name and what am I studying?")]},
    config=config
)
print("Memory Recall Response:", turn2["messages"][-1].content)

### 🌊 6. Streaming Execution Modes (`stream` & `stream_mode`)
- **`stream_mode="updates"`**: Har node ke state update ko emit karta hai (best for debugging & step tracking).
- **`stream_mode="values"`**: Har step par complete graph state emit karta hai.

In [6]:
query = {"messages": [HumanMessage(content="What is 45 multiplied by 89?")]}
stream_config = {"configurable": {"thread_id": "math_stream_1"}}

print("Streaming node execution updates:")
for chunk in stateful_graph.stream(query, config=stream_config, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"👉 Node [{node_name}] produced update.")

### 🛑 7. Human-in-the-Loop (HITL) Interrupts
- **Parameter**: `interrupt_before=["tools"]`
- **Purpose**: Tool execute hone se pehle graph ko pause karna taaki human approval ya verification ho sake.

In [7]:
hitl_memory = MemorySaver()
hitl_graph = tool_builder.compile(
    checkpointer=hitl_memory,
    interrupt_before=["tools"]  # Pauses execution before tool executes
)

hitl_config = {"configurable": {"thread_id": "hitl_thread_1"}}
hitl_input = {"messages": [HumanMessage(content="Calculate 125 multiplied by 8.")]}

# Step 1: Run until tool interrupt
for event in hitl_graph.stream(hitl_input, config=hitl_config):
    pass

current_state = hitl_graph.get_state(hitl_config)
print("Next action pending approval:", current_state.next)
print("Tool call to be approved:", current_state.values["messages"][-1].tool_calls)

# Step 2: Human approves by resuming with None input
for event in hitl_graph.stream(None, config=hitl_config):
    pass

final_hitl_state = hitl_graph.get_state(hitl_config)
print("Final result after approval:", final_hitl_state.values["messages"][-1].content)